In [102]:
import math
import random
import hashlib
import json

# ICC132 -Actividad integradora: implementación de un esquema asimétrico de firma y verificación

## Codigo Fuente

### Funciones basicas

In [120]:
# FUNCIONES DE LA ACTIVIDAD ANTERIOR

#Generacion de llaves
def ifPrimo(num):
    if num <= 1:
        return False
    for i in range(2, int(math.sqrt(num)) + 1):
        if num % i == 0:
            return False  
    return True

def numPrimo(minVal, maxVal):
    primo = random.randint(minVal, maxVal)
    valid = ifPrimo(primo)
    while valid == False:
        primo = random.randint(minVal, maxVal)
        valid = ifPrimo(primo)
    return primo

def pqDist(min, max): 
    p = numPrimo(min, max)
    q = numPrimo(min, max)
    while p == q:
        q = numPrimo(min, max)
    return p,q

def llaveRSA(min,max):
    p,q = pqDist(min, max)
    n = p * q 
    euler = (p - 1) * (q - 1)
    e = random.randrange(2, euler)
    while math.gcd(e, euler) != 1:
        e = random.randrange(2, euler)
    d = pow(e, -1, euler) 
    public = (e,n)
    private = (n,d)
    return public, private

def hashing(texto):
    texto = str(texto)
    hash = hashlib.sha256(texto.encode('utf-8'))
    hex = hash.hexdigest()    
    numEntero = int(hex, 16)
    return numEntero

#Firmas

def signRSA(texto, llavePriv):
    n , d = llavePriv
    entero = hashing(texto)
    cutEntero = entero % n 
    signed = pow(cutEntero, d, n)
    return signed

def verif(texto, sign, llavePublic):
    try: 
        e , n = llavePublic
        entero = hashing(texto)
        cutEntero = entero % n
        signedText = pow(sign, e , n)
        if (cutEntero == signedText):
            return True
        else:
            return False
    except Exception as e:
        print("Error de formato al verificar")
        return False


In [121]:
#FUNCIONES NUEVAS

#Cifrado

def cifrarRSA(entero, llavePublica):
    e, n = llavePublica
    cifrado = pow(entero, e, n)
    return cifrado

def descifrarRSA(numeroCifrado, llavePrivada):
    n, d = llavePrivada
    descifrado = pow(numeroCifrado, d, n)
    return descifrado


def generarClaveSesion():
    return random.randint(100000, 999999)

def cifrarMensaje(texto, claveSesion):
    mensaje_cifrado = []
    for letra in texto:
        valor_oculto = ord(letra) + claveSesion
        mensaje_cifrado.append(valor_oculto)
    return mensaje_cifrado

def descifrarMensaje(listaCifrada, claveSesion):
    texto_original = ""
    for numero in listaCifrada:
        letra = chr(numero - claveSesion)
        texto_original += letra
    return texto_original

# generar usuarios
def generarUsuario(nombre):
    public, private = llaveRSA(100000, 999999)
    return {
        "nombre": nombre,
        "publica": public,
        "privada": private
    }

#JSON
def empaquetarJSON(emisor, receptor, msgCifrado, claveCifrada, firma, pubKeyEmisor):
    paquete = {
        "sender": emisor,
        "receiver": receptor,
        "encrypted_message": msgCifrado,
        "encrypted_session_key": claveCifrada,
        "signature": firma,
        "hash_algorithm": "SHA-256",
        "sender_public_key": {
            "e": pubKeyEmisor[0],
            "n": pubKeyEmisor[1]
        }
    }
    nombreJSON = f"mensaje_{emisor}_a_{receptor}.json"
    with open(nombreJSON, 'w') as archivo:
        json.dump(paquete, archivo, indent=4) 
    return nombreJSON

def cargarJSON(nombre_archivo):
    with open(nombre_archivo, 'r') as archivo:
        return json.load(archivo)


### Funciones principales

In [122]:
#Creacion del mensaje
def flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio):
    print(f"\n[EMISOR] {nombreEmisor} está preparando un mensaje...")
    
    # Validamos si el mensaje está vacío (Caso 2)
    if mensageOriginal == "":
        print("[ADVERTENCIA] Se está enviando un mensaje vacío.")
        
    print(f"Texto a enviar: '{mensageOriginal}'")

    try:
        # generar y cifrar mensaje
        claveSesion = generarClaveSesion()
        mensajeCifrado = cifrarMensaje(mensageOriginal, claveSesion)

        # cifrar la clave de sesion usando la llave del receptor
        llavePubReceptor = directorio[nombreReceptor]["publica"]
        claveSesionCifrada = cifrarRSA(claveSesion, llavePubReceptor)

        # firmar mensaje usando llave del emisor
        llavePrivEmisor = directorio[nombreEmisor]["privada"]
        firmaDigital = signRSA(mensageOriginal, llavePrivEmisor)

        # guardar JSON
        llavePubEmisor = directorio[nombreEmisor]["publica"]
        archivo = empaquetarJSON(nombreEmisor, nombreReceptor, mensajeCifrado, claveSesionCifrada, firmaDigital, llavePubEmisor)
        print(f"[SISTEMA] Archivo '{archivo}' creado con éxito.")
        
        return archivo
        
    except Exception as e:
        print(f"[SISTEMA] ERROR CRÍTICO al generar el paquete: {e}")
        return None

#recepcion del mensaje
def flujoReceptor(nombreReceptor, archivo, directorio):
    print(f"\n[RECEPTOR] {nombreReceptor} ha recibido el archivo JSON '{archivo}'")

    try:
        # cargar JSON
        paqueteRecibido = cargarJSON(archivo)
        
        # Validar que el paquete no esté incompleto (Caso 7)
        camposRequeridos = ["sender", "receiver", "encrypted_message", "encrypted_session_key", "signature", "sender_public_key"]
        for campo in camposRequeridos:
            if campo not in paqueteRecibido:
                print(f"[RESULTADO] EL PAQUETE FUE RECHAZADO: Faltan datos esenciales (Falta el campo '{campo}').")
                return False, None

        # descifrar clave de sesion usando llave privada del receptor
        llavePrivReceptor = directorio[nombreReceptor]["privada"]
        claveRecuperada = descifrarRSA(paqueteRecibido["encrypted_session_key"], llavePrivReceptor)

        # descifrar Mensaje (Aquí atrapa el Caso 6 y Caso 8 si los datos son basura)
        textoDescifrado = descifrarMensaje(paqueteRecibido["encrypted_message"], claveRecuperada)

        # verificar Firma usando llave publica del emisor 
        llavePubEmisorPaquete = (paqueteRecibido["sender_public_key"]["e"], paqueteRecibido["sender_public_key"]["n"])
        firmaRecibida = paqueteRecibido["signature"]

        valido = verif(textoDescifrado, firmaRecibida, llavePubEmisorPaquete)

        # resultados finales
        print("--------------------------------------------------")
        if valido:
            print("[RESULTADO] La firma es VALIDA. El mensaje no fue alterado y corresponde al emisor indicado.")
            print(f"[MENSAJE RECUPERADO]: '{textoDescifrado}'")
        else:
            print("[RESULTADO] La firma NO ES VALIDA. El mensaje pudo haber sido alterado o la llave pública no corresponde al emisor.")
        print("--------------------------------------------------")
        
        return valido, textoDescifrado

    except KeyError as e:
        print(f"[RESULTADO] EL PAQUETE FUE RECHAZADO: Estructura del JSON corrupta o llaves faltantes. Detalles: {e}")
        return False, None
    except Exception as e:
        print(f"[RESULTADO] EL PAQUETE FUE RECHAZADO: Error al procesar los datos (posible ataque, alteración o llave incorrecta).")
        return False, None

### Flujo normal

In [123]:
print("[SISTEMA] Creacion de directorio y usuarios")

#creacion de usuarios 
nombreEmisor = "Pablo"
nombreReceptor = "Miguela"

print(f"Nombre del Emisor: {nombreEmisor}")
print(f"Nombre del Receptor: {nombreReceptor}")

directorio = {
    nombreEmisor: generarUsuario(nombreEmisor),
    nombreReceptor: generarUsuario(nombreReceptor)
}

mensageOriginal = f"Hola {nombreReceptor}, este es un mensaje super secreto!"

# se crea el mensaje y se manda como un JSON 
archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

# se recibe el JSON y se verifica
esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)

[SISTEMA] Creacion de directorio y usuarios
Nombre del Emisor: Pablo
Nombre del Receptor: Miguela

[EMISOR] Pablo está preparando un mensaje...
Texto a enviar: 'Hola Miguela, este es un mensaje super secreto!'
[SISTEMA] Archivo 'mensaje_Pablo_a_Miguela.json' creado con éxito.

[RECEPTOR] Miguela ha recibido el archivo JSON 'mensaje_Pablo_a_Miguela.json'
--------------------------------------------------
[RESULTADO] La firma es VALIDA. El mensaje no fue alterado y corresponde al emisor indicado.
[MENSAJE RECUPERADO]: 'Hola Miguela, este es un mensaje super secreto!'
--------------------------------------------------


## Documento de diseño

### 1. Descripción general del sistema

El sistema desarrollado es una simulación en Python de comunicación segura entre múltiples usuarios. Garantiza tres principios fundamentales de la criptografía: confidencialidad: solo el destinatario puede leer el mensaje, integridad: el mensaje no puede ser alterado en el camino y autenticidad: se comprueba la identidad del emisor. Para lograrlo, utiliza un esquema híbrido que combina operaciones matemáticas basadas en el algoritmo asimétrico RSA puro, funciones de hash y un cifrado de desplazamiento.


### 2. Flujo de envío y recepción del mensaje

El ciclo de vida del mensaje se divide en dos fases principales:

- **Flujo del Emisor:** El usuario redacta un mensaje en texto plano. El sistema genera una clave de sesión aleatoria, cifra el mensaje con ella, y luego protege esa clave de sesión usando la llave pública del receptor. Posteriormente, se calcula un Hash del mensaje original y se firma con la llave privada del emisor. Toda esta información se empaqueta en un archivo JSON.

- **Flujo del Receptor:** El destinatario abre el archivo JSON. Primero, utiliza su llave privada para recuperar la clave de sesión y así descifrar el texto oculto. Luego, toma ese texto recuperado y verifica matemáticamente la firma digital adjunta utilizando la llave pública del emisor. Si las matemáticas coinciden, el mensaje es validado; si no, el paquete es rechazado.


### 3. Explicación de las llaves RSA

El sistema no utiliza librerías criptográficas externas, sino matemáticas puras. Las llaves se generan seleccionando dos números primos aleatorios ($p$ y $q$) a través de funciones propias de validación de primalidad. Se calcula el módulo $n = p \times q$ y la indicatriz de Euler $\phi(n)$. Luego se elige un exponente público $e$ (coprimo de $\phi(n)$) y se calcula su inverso multiplicativo $d$.La llave pública está formada por la tupla (e, n). Esta se comparte y se usa para cifrar datos dirigidos a ese usuario o para verificar sus firmas.La llave privada está formada por la tupla (n, d). Se mantiene secreta y se usa para descifrar datos recibidos y firmar mensajes salientes.



### 4. Explicación del cifrado utilizado

Para el cifrado de datos, el sistema implementa un enfoque híbrido:

- **Representación interna y cifrado del texto (Simétrico):** El mensaje original no se cifra directamente con RSA por cuestiones de rendimiento. En su lugar, el texto se convierte internamente en un arreglo numérico. A cada carácter se le extrae su valor ASCII y se le suma matemáticamente una clave de sesión (un número aleatorio de 4 dígitos).

- **Cifrado de la clave (Asimétrico):** Para proteger esta clave de sesión, se aplica RSA puro (pow(clave, e, n)) utilizando la llave pública del receptor.


### 5. Explicación de la firma digital

Para garantizar la integridad y autenticidad, no se firma el mensaje completo, sino su hash. Se toma el texto original y se pasa por el algoritmo SHA-256 (vía la librería hashlib), generando un número entero único. Este número se trunca con el módulo $n$ y luego se eleva a la potencia de la llave privada del emisor ($d$) utilizando aritmética modular. El resultado es la firma digital que viaja en el paquete.

### 6. Qué estructura tiene el paquete JSON

Para empaquetar la comunicación, se definió un diccionario estricto que posteriormente se serializa como un archivo .json. La estructura garantiza que viaje toda la información necesaria para la verificación sin exponer las llaves privadas:

- **sender (String):** Nombre del emisor.

- **receiver (String):** Nombre del receptor esperado.

- **encrypted_message (Array de Enteros):** El texto ofuscado mediante la clave de sesión.

- **encrypted_session_key (Entero):** La clave de sesión protegida con RSA.

- **signature (Entero):** La firma digital generada a partir del Hash SHA-256.

- **hash_algorithm (String):** Constante indicando "SHA-256".

- **sender_public_key (Objeto):** Contiene los valores "e" y "n" públicos del emisor.

### 7. Validaciones implementadas

El sistema (flujoReceptor) está blindado mediante bloques try...except para manejar distintos vectores de error sin colapsar:

- Verifica que el JSON no esté incompleto, validando la existencia de todas las claves requeridas (ej. detecta si falta la firma).

- Maneja con gracia datos mal formados o llaves incorrectas que ocasionarían errores matemáticos en el proceso de descifrado.

- La función principal verif recalcula el hash del mensaje recibido y evalúa matemáticamente si corresponde con la firma. Si un intruso altera una sola letra del mensaje o un dígito de la firma, la ecuación devuelve False y el sistema arroja una alerta rechazando el paquete.

### 8. Limitaciones y observaciones técnicas

Por tratarse de una implementación educativa, el sistema presenta limitaciones deliberadas que serían vulnerabilidades en el mundo real:
- **Tamaño de llaves:** Se usaron primos pequeños (1000 a 9999) para evitar lentitud en Python. En la realidad, esto permitiría factorizar $n$ en milisegundos mediante fuerza bruta.
- **Cifrado simétrico débil:** El desplazamiento ASCII utilizado para el mensaje es trivial de romper mediante análisis de frecuencias. Un sistema real usaría algoritmos robustos como AES.
- **Falta de Padding:** La implementación pura de RSA no incluye esquemas de relleno (como OAEP), lo que hace al cifrado determinista y susceptible a ataques de texto cifrado elegido.

## Preguntas

### 1. ¿Qué parte del código corresponde a la generación de llaves RSA?

La generación de llaves ocurre en la función llaveRSA. Aquí es donde el sistema toma dos primos aleatorios, calcula el módulo $n$, la indicatriz de Euler, y deriva los exponentes público (e) y privado (d).

```python
def llaveRSA(min, max):
    p, q = pqDist(min, max)
    n = p * q
    euler = (p - 1) * (q - 1)

    e = random.randrange(2, euler)
    while math.gcd(e, euler) != 1:
        e = random.randrange(2, euler)

    d = pow(e, -1, euler)

    public = (e, n)
    private = (n, d)

    return public, private
```

Esto se invoca para cada participante dentro de la función generarUsuario.

### 2. ¿Qué parte corresponde al cifrado del mensaje?

El sistema utiliza un cifrado híbrido, por lo que el proceso se divide en dos bloques. Primero, el texto se ofusca sumándole una clave de sesión aleatoria a cada carácter

```python
def cifrarMensaje(texto, claveSesion):
    mensaje_cifrado = []
    for letra in texto:
        valor_oculto = ord(letra) + claveSesion
        mensaje_cifrado.append(valor_oculto)
    return mensaje_cifrado
```
Y luego, para proteger esa claveSesion en su viaje, se cifra utilizando matemáticamente la llave pública del receptor:
```python
def cifrarRSA(entero, llavePublica):
    e, n = llavePublica
    cifrado = pow(entero, e, n)
    return cifrado
```


### 3. ¿Qué parte corresponde a la firma digital?

La firma digital se compone de dos pasos. Primero, la función hashing convierte el texto original en una huella digital única (SHA-256) y la transforma en un número entero. Luego, la función signRSA cifra ese número utilizando la llave privada del emisor:

```python
def signRSA(texto, llavePriv):
    n , d = llavePriv
    entero = hashing(texto)
    cutEntero = entero % n 
    signed = pow(cutEntero, d, n)
    return signed
```

### 4. ¿Qué ocurre si el mensaje se modifica después de generar la firma?

Si un intruso altera el mensaje como vimos en la Prueba 3, al pasarlo por la función verif, el sistema calculará el Hash del mensaje hackeado cutEntero y lo comparará con la firma original descifrada signedText. Como no coincidirán, la función devolverá False.

```python
def verif(texto, sign, llavePublic):
    # ... (código omitido) ...
        entero = hashing(texto)
        cutEntero = entero % n
        signedText = pow(sign, e , n)
        if (cutEntero == signedText):
            return True
        else:
            return False # <--- Caerá aquí
```
Y en el flujo del receptor regresaria NO VALIDA
```python
if valido:
            print("[RESULTADO] La firma es VALIDA...")
        else:
            print("[RESULTADO] La firma NO ES VALIDA. El mensaje pudo haber sido alterado o la llave pública no corresponde al emisor.")
```

### 5. ¿Qué ocurre si se intenta descifrar con la llave privada incorrecta?
Al usar una llave privada errónea como en la Prueba 6, la matemática de la función descifrarRSA no generará un error de inmediato, sino que devolverá una "clave de sesión basura" gigante. Cuando tu programa intente usar esa clave basura en descifrarMensaje, arrojará un texto ilegible o fallará al procesar los caracteres.

El código está protegido contra este colapso en la función flujoReceptor gracias al último bloque except:

```python
except Exception as e:
        print(f"[RESULTADO] EL PAQUETE FUE RECHAZADO: Error al procesar los datos (posible ataque, alteración o llave incorrecta).")
        return False, None
```


### 6. ¿Cuál es la principal limitación de la implementación?

La mayor vulnerabilidad de el código está en la asignacion como valores minimos y maximos, por razones de demostraciones, se mantienen los valores bajos y en una situacion de seguridad estos numeros serian faciles de descifrar por fuerza bruta. Además, la función cifrarMensaje es solo un desplazamiento (tipo cifrado César) en lugar de usar un estándar robusto como AES, lo que lo hace vulnerable a un ataque de análisis de frecuencias.

## Bitácora de desarrollo


| Etapa | Problema Encontrado | Decisión Tomada | Ajuste Realizado | Explicación de la Decisión |
| :--- | :--- | :--- | :--- | :--- |
| **Estructuración de Datos (Gestión de Usuarios)** | Administrar múltiples variables o arreglos sueltos provocaba confusiones de indexación, resultando en errores donde se usaba la llave pública equivocada para cifrar o verificar. | Centralizar la gestión de identidades migrando de variables sueltas a un diccionario general. | Se implementó la estructura `directorio` donde la clave es el nombre y los valores son sus llaves (ej. `directorio["Miguela"]["privada"]`). | Los diccionarios (pares clave-valor) permiten un mapeo explícito e intuitivo. Esto eliminó errores de asignación, hizo escalable el sistema para $N$ usuarios y permitió referenciar participantes usando variables dinámicas. |
| **Validación de Paquetes JSON** | Al simular el caso de prueba de "JSON incompleto" (borrando la firma), el programa intentaba buscar el valor faltante y colapsaba lanzando una excepción `KeyError`. | Validar la integridad estructural del paquete de forma preventiva antes de realizar cálculos criptográficos. | Se añadió una lista de `camposRequeridos` y bloques `try...except KeyError` dentro de la función `flujoReceptor`. | En un entorno real, los paquetes pueden llegar mal formados o interceptados. El sistema debía ser capaz de detectar faltantes para rechazar el archivo limpiamente sin detener la ejecución de todo el programa. |
| **Modularización del Flujo Principal** | El código original se ejecutaba de forma lineal y global. Probar los 9 escenarios requeridos implicaba copiar y pegar el mismo bloque de código repetidamente, lo que ensuciaba el script y dificultaba el testing. | Encapsular la lógica de cifrado/empaquetado y recepción/verificación en funciones independientes y reutilizables. | Se agruparon las operaciones en dos funciones principales: `flujoEmisor` y `flujoReceptor`, pasando los datos (usuarios, mensaje, directorio) como parámetros. | Convertir el código lineal en funciones permitió ejecutar los 9 casos de validación de forma automatizada llamando a las mismas funciones con distintos datos. Esto hizo el código más limpio, mantenible y fácil de probar los casos. |
| **Generación de Entregables (Sobrescritura)** | Al ejecutar los 9 escenarios de prueba de forma automatizada, el archivo `mensaje.json` se sobrescribía constantemente, perdiendo la evidencia de las pruebas anteriores. | Desviar la salida de los archivos de prueba sin modificar la lógica interna ni los parámetros de las funciones originales. | Se crearon pares de usuarios adicionales y "desechables" (ej. Carlos/Diana, Luis/Carmen) específicos para cada escenario requerido. | Agregar parámetros opcionales a la función original hubiera tomado mas tiempo. Al crear identidades distintas, el código demostró su capacidad para manejar múltiples sesiones, generando naturalmente archivos con nombres únicos para los entregables. |

## Verificaciones

### Creacion de usuarios

In [124]:
print("[SISTEMA] Creacion de directorio y usuarios")

directorio = {
    #Nombres genericos
    "Pancho": generarUsuario("Pablo"),
    "Pancha": generarUsuario("Miguela"),
    
    #Mensaje valido (Prueba 1)
    "Pablo": generarUsuario("Pablo"),
    "Miguela": generarUsuario("Miguela"),
    
    #Mensaje alterado (Prueba 3)
    "Carlos": generarUsuario("Carlos"),
    "Diana": generarUsuario("Diana"),
    
    #Paquete incompleto (Prueba 7)
    "Luis": generarUsuario("Luis"),
    "Carmen": generarUsuario("Carmen"),
    
    #Usuario Intruso
    "Intruso": generarUsuario("Intruso") 
}

[SISTEMA] Creacion de directorio y usuarios


### Caso 1. mensaje válido enviado del emisor al receptor;

In [125]:
nombreEmisor = "Pablo"
nombreReceptor = "Miguela"
mensageOriginal = f"Hola {nombreReceptor}, este es un mensaje valido."

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)
esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Pablo está preparando un mensaje...
Texto a enviar: 'Hola Miguela, este es un mensaje valido.'
[SISTEMA] Archivo 'mensaje_Pablo_a_Miguela.json' creado con éxito.

[RECEPTOR] Miguela ha recibido el archivo JSON 'mensaje_Pablo_a_Miguela.json'
--------------------------------------------------
[RESULTADO] La firma es VALIDA. El mensaje no fue alterado y corresponde al emisor indicado.
[MENSAJE RECUPERADO]: 'Hola Miguela, este es un mensaje valido.'
--------------------------------------------------


### Caso 2. mensaje vacío;

In [126]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

mensageOriginal = ""
archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)
esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Pancha está preparando un mensaje...
[ADVERTENCIA] Se está enviando un mensaje vacío.
Texto a enviar: ''
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
--------------------------------------------------
[RESULTADO] La firma es VALIDA. El mensaje no fue alterado y corresponde al emisor indicado.
[MENSAJE RECUPERADO]: ''
--------------------------------------------------


### Caso 3. mensaje alterado después de haber sido firmado;

In [127]:
nombreEmisor = "Carlos"
nombreReceptor = "Diana"
mensageOriginal = "Mensaje original secreto que será alterado"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

#Abrimos el JSON y agregamos un numero extra al mensaje cifrado
with open(archivoGenerado, 'r') as f:
    datos_hackeados = json.load(f)
datos_hackeados["encrypted_message"].append(9999) 
with open(archivoGenerado, 'w') as f:
    json.dump(datos_hackeados, f)

esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Carlos está preparando un mensaje...
Texto a enviar: 'Mensaje original secreto que será alterado'
[SISTEMA] Archivo 'mensaje_Carlos_a_Diana.json' creado con éxito.

[RECEPTOR] Diana ha recibido el archivo JSON 'mensaje_Carlos_a_Diana.json'
[RESULTADO] EL PAQUETE FUE RECHAZADO: Error al procesar los datos (posible ataque, alteración o llave incorrecta).


### Caso 4. firma alterada manualmente dentro del JSON

In [128]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

# modificamos el uúmero de la firma sumandole 1
with open(archivoGenerado, 'r') as f:
    datos_hackeados = json.load(f)
datos_hackeados["signature"] = datos_hackeados["signature"] + 1 
with open(archivoGenerado, 'w') as f:
    json.dump(datos_hackeados, f)

esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Mensaje original secreto que será alterado'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
--------------------------------------------------
[RESULTADO] La firma NO ES VALIDA. El mensaje pudo haber sido alterado o la llave pública no corresponde al emisor.
--------------------------------------------------


### Caso 5. Intento de verificación con una llave pública incorrecta;

In [129]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

# Sustituimos la llave publica del emisor por la del Intruso en el JSON
with open(archivoGenerado, 'r') as f:
    datos_hackeados = json.load(f)
datos_hackeados["sender_public_key"]["e"] = directorio[nombreIntruso]["publica"][0]
datos_hackeados["sender_public_key"]["n"] = directorio[nombreIntruso]["publica"][1]
with open(archivoGenerado, 'w') as f:
    json.dump(datos_hackeados, f)

esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Mensaje original secreto que será alterado'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
--------------------------------------------------
[RESULTADO] La firma NO ES VALIDA. El mensaje pudo haber sido alterado o la llave pública no corresponde al emisor.
--------------------------------------------------


### Caso 6. Intento de descifrado con la llave privada de otro usuario

In [130]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

# le quitamos temporalmente su llave a Miguela y le damos la del Intruso
llaveOriginal = directorio[nombreReceptor]["privada"]
directorio[nombreReceptor]["privada"] = directorio[nombreIntruso]["privada"]

# descifrar con la llave incorrecta que le asignamos
esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)

directorio[nombreReceptor]["privada"] = llaveOriginal 


[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Mensaje original secreto que será alterado'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
[RESULTADO] EL PAQUETE FUE RECHAZADO: Error al procesar los datos (posible ataque, alteración o llave incorrecta).


### Caso 7. Paquete JSON incompleto

In [131]:
nombreEmisor = "Luis"
nombreReceptor = "Carmen"

mensageOriginal = "Este mensaje perderá su firma en el camino"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

#Borramos por completo el campo de la firma del archivo JSON
with open(archivoGenerado, 'r') as f:
    datos_hackeados = json.load(f)
del datos_hackeados["signature"] 
with open(archivoGenerado, 'w') as f:
    json.dump(datos_hackeados, f)

esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Luis está preparando un mensaje...
Texto a enviar: 'Este mensaje perderá su firma en el camino'
[SISTEMA] Archivo 'mensaje_Luis_a_Carmen.json' creado con éxito.

[RECEPTOR] Carmen ha recibido el archivo JSON 'mensaje_Luis_a_Carmen.json'
[RESULTADO] EL PAQUETE FUE RECHAZADO: Faltan datos esenciales (Falta el campo 'signature').


### Caso 8. Paquete JSON con datos mal formados

In [132]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

archivoGenerado = flujoEmisor(nombreEmisor, nombreReceptor, mensageOriginal, directorio)

# cambiamos la lista de numeros por un texto simple que el programa no espera
with open(archivoGenerado, 'r') as f:
    datos_hackeados = json.load(f)
datos_hackeados["encrypted_message"] = "Esto debería ser una lista, no texto" 
with open(archivoGenerado, 'w') as f:
    json.dump(datos_hackeados, f)

esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoGenerado, directorio)


[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Este mensaje perderá su firma en el camino'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
[RESULTADO] EL PAQUETE FUE RECHAZADO: Error al procesar los datos (posible ataque, alteración o llave incorrecta).


### Caso 9. Firma que no corresponde al mensaje recibido.

In [133]:
nombreEmisor = "Pancha"
nombreReceptor = "Pancho"

# Generamos dos mensajes completamente diferentes
archivoA = flujoEmisor(nombreEmisor, nombreReceptor, "Pagar $100 dolares a Pablo", directorio)
with open(archivoA, 'r') as f:
    datosA = json.load(f)

archivoB = flujoEmisor(nombreEmisor, nombreReceptor, "Pagar $5000 dolares a Pablo", directorio)
with open(archivoB, 'r') as f:
    datosB = json.load(f)

# Tomamos la firma valida del archivo B y la pegamos en el archivo A
datosA["signature"] = datosB["signature"]
with open(archivoA, 'w') as f:
    json.dump(datosA, f)

# Enviamos el archivo A modificado al receptor
esValido, mensajeFinal = flujoReceptor(nombreReceptor, archivoA, directorio)


[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Pagar $100 dolares a Pablo'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[EMISOR] Pancha está preparando un mensaje...
Texto a enviar: 'Pagar $5000 dolares a Pablo'
[SISTEMA] Archivo 'mensaje_Pancha_a_Pancho.json' creado con éxito.

[RECEPTOR] Pancho ha recibido el archivo JSON 'mensaje_Pancha_a_Pancho.json'
--------------------------------------------------
[RESULTADO] La firma NO ES VALIDA. El mensaje pudo haber sido alterado o la llave pública no corresponde al emisor.
--------------------------------------------------
